# 🏎️ Fabric Racing Game v3 - Real-Time Telemetry

Telemetria inviata **automaticamente in tempo reale** mentre giochi!

## Come Giocare (2 Step)
1. **Run Cell 1** → Configura credenziali Eventstream e genera SAS Token
2. **Run Cell 2** → Gioca! Eventi inviati ogni 2 secondi automaticamente

## Dove trovo le credenziali?
Eventstream → Custom Endpoint → Keys → copia i 4 valori

In [ ]:
# ⚙️ CELL 1: CONFIGURA CREDENZIALI EVENTSTREAM
# Copia i valori da: Eventstream → Custom Endpoint → Keys

EH_NS = "YOUR_NAMESPACE.servicebus.windows.net"  # Event Hub namespace
EH_NAME = "es_XXXXX"                              # Event Hub name  
EH_KEY_NAME = "key_XXXXX"                         # Shared access key name
EH_KEY = "YOUR_KEY_VALUE"                         # Shared access key

PLAYER_NAME = "Player1"  # Il tuo nome pilota

# Genera SAS Token (valido 4 ore)
import hmac, hashlib, base64, urllib.parse, time

uri = f'https://{EH_NS}/{EH_NAME}'
ttl = str(int(time.time()) + 14400)  # 4 ore
eu = urllib.parse.quote_plus(uri)
sts = (eu + '\n' + ttl).encode('utf-8')
sig = base64.b64encode(hmac.HMAC(EH_KEY.encode('utf-8'), sts, hashlib.sha256).digest()).decode()
SAS_TOKEN = f'SharedAccessSignature sr={eu}&sig={urllib.parse.quote_plus(sig)}&se={ttl}&skn={EH_KEY_NAME}'
ES_URL = f"https://{EH_NS}/{EH_NAME}/messages?timeout=60&api-version=2014-01"

print("✅ SAS Token generato (valido 4 ore)")
print(f"🎮 Pilota: {PLAYER_NAME}")
print("\n▶️ Esegui Cell 2 per giocare!")

In [ ]:
# 🎮 CELL 2: GIOCA! (Telemetria automatica ogni 2 sec)
from IPython.display import display, HTML
import uuid, json

session_id = str(uuid.uuid4())

# Bridge JavaScript per inviare telemetria via SAS
bridge_js = f'''
<script>
(function(){{
    var url = '{ES_URL}';
    var sas = '{SAS_TOKEN}';
    var buf = [], sent = 0, errs = 0;

    window.addEventListener('gev', function(e) {{ buf.push(e.detail); }});

    setInterval(function() {{
        if (buf.length === 0) return;
        var batch = buf.slice(); buf = [];
        batch.forEach(function(ev) {{
            fetch(url, {{
                method: 'POST',
                headers: {{'Authorization': sas, 'Content-Type': 'application/json'}},
                body: JSON.stringify(ev)
            }})
            .then(function(r) {{
                if (r.status === 201) {{ sent++; if(window.__setTelSt) window.__setTelSt('live'); }}
                else {{ errs++; if(window.__setTelSt) window.__setTelSt('err'); }}
            }})
            .catch(function() {{ errs++; if(window.__setTelSt) window.__setTelSt('err'); }});
        }});
    }}, 2000);

    window.addEventListener('gend', function(e) {{
        if (buf.length > 0) {{
            buf.forEach(function(ev) {{
                fetch(url, {{
                    method: 'POST',
                    headers: {{'Authorization': sas, 'Content-Type': 'application/json'}},
                    body: JSON.stringify(ev)
                }});
            }});
            buf = [];
        }}
        if(window.__setTelSt) window.__setTelSt('saved');
    }});

    if(window.__setTelSt) window.__setTelSt('live');
    console.log('TelemetryBridge: connected to Eventstream');
}})();
</script>
'''

game_html = f'''
<!DOCTYPE html>
<html>
<head>
<style>
* {{ margin: 0; padding: 0; box-sizing: border-box; }}
#game-wrapper {{
    width: 100%; display: flex; flex-direction: column; align-items: center;
    font-family: 'Segoe UI', Arial, sans-serif;
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
    padding: 20px; border-radius: 15px;
}}
#game-container {{
    position: relative; width: 600px; height: 500px;
    background: #2d3436; border-radius: 10px; overflow: hidden;
    box-shadow: 0 10px 30px rgba(0,0,0,0.5);
}}
canvas {{ display: block; }}
#hud {{
    position: absolute; top: 10px; left: 10px; right: 50px;
    display: flex; justify-content: space-between;
    color: white; font-size: 14px; text-shadow: 2px 2px 4px rgba(0,0,0,0.8); z-index: 10;
}}
#hud-left, #hud-right {{ background: rgba(0,0,0,0.7); padding: 10px 14px; border-radius: 8px; }}
#lives {{ color: #ff6b6b; font-size: 18px; letter-spacing: 3px; }}
#progress-bar {{
    position: absolute; right: 15px; top: 80px; bottom: 80px; width: 25px;
    background: rgba(0,0,0,0.7); border-radius: 12px; z-index: 10;
}}
#progress-fill {{ position: absolute; bottom: 0; width: 100%; background: linear-gradient(to top, #00b894, #55efc4); border-radius: 10px; transition: height 0.15s; }}
#progress-car {{ position: absolute; left: -8px; width: 40px; text-align: center; font-size: 18px; transition: bottom 0.15s; }}
#start-btn {{
    position: absolute; top: 50%; left: 50%; transform: translate(-50%, -50%);
    padding: 25px 60px; font-size: 32px;
    background: linear-gradient(135deg, #00b894, #00a085);
    color: white; border: none; border-radius: 20px; cursor: pointer; z-index: 100;
}}
#level-info, #game-over {{
    position: absolute; top: 50%; left: 50%; transform: translate(-50%, -50%);
    background: rgba(0,0,0,0.95); padding: 40px; border-radius: 15px;
    color: white; text-align: center; z-index: 100; display: none; min-width: 320px;
}}
#telemetry-status {{ position: absolute; bottom: 5px; right: 5px; font-size: 10px; padding: 2px 6px; border-radius: 4px; }}
#event-count {{ position: absolute; bottom: 5px; left: 5px; font-size: 11px; color: #888; }}
.multiplier {{ color: #ffd700; }}
#leaderboard {{ margin-top: 15px; background: rgba(255,255,255,0.08); padding: 15px 20px; border-radius: 10px; width: 600px; box-sizing: border-box; }}
#leaderboard h3 {{ color: #ffd700; margin-bottom: 10px; font-size: 16px; }}
#leaderboard-list {{ color: white; list-style: none; padding: 0; margin: 0; font-size: 13px; }}
#leaderboard-list li {{ padding: 5px 0; border-bottom: 1px solid rgba(255,255,255,0.1); display: flex; justify-content: space-between; }}
#leaderboard-list li:last-child {{ border-bottom: none; }}
</style>
</head>
<body>
<div id="game-wrapper" tabindex="0">
    <div id="game-container">
        <canvas id="gameCanvas" width="600" height="500"></canvas>
        <div id="hud">
            <div id="hud-left">
                <div style="font-size:16px; color:#ffd700;">🏎️ <span id="level-name">Lakehouse Lane</span></div>
                <div>Level: <span id="level">1</span>/10</div>
                <div id="lives">❤️❤️❤️</div>
            </div>
            <div id="hud-right">
                <div style="font-size:18px;">Score: <span id="score">0</span></div>
                <div>Multiplier: <span id="multiplier" class="multiplier">x1</span></div>
                <div>Target: <span id="target" style="color:#ffd700;">500</span></div>
            </div>
        </div>
        <div id="progress-bar">
            <div id="progress-fill" style="height: 0%"></div>
            <div id="progress-car">🏎️</div>
        </div>
        <button id="start-btn" onclick="startGame()">▶ START</button>
        <div id="level-info">
            <h2 id="level-title">Level Complete!</h2>
            <p id="level-message"></p>
            <button id="continue-btn" onclick="continueGame()" style="margin-top:15px; padding:12px 30px; background:#00b894; color:white; border:none; border-radius:10px; cursor:pointer;">Continue</button>
        </div>
        <div id="game-over"></div>
        <div id="telemetry-status" style="background:#27ae60; color:white;">📡 LIVE</div>
        <div id="event-count">Events: 0</div>
    </div>
    <div id="leaderboard"><h3>🏆 Best Scores</h3><ol id="leaderboard-list"></ol></div>
</div>

<script>
window.gameEvents = [];
var telSt = 'live';
window.__setTelSt = function(s) {{
    telSt = s;
    var el = document.getElementById('telemetry-status');
    if (s === 'live') {{ el.style.background = '#27ae60'; el.textContent = '📡 LIVE'; }}
    else if (s === 'saved') {{ el.style.background = '#3498db'; el.textContent = '✅ SAVED'; }}
    else {{ el.style.background = '#e74c3c'; el.textContent = '⚠️ OFFLINE'; }}
}};

const LEVELS = [
    {{ name: "Lakehouse Lane", target: 500, stars: 12, bugs: 5, length: 1500, speed: 4 }},
    {{ name: "Pipeline Pass", target: 800, stars: 14, bugs: 7, length: 1800, speed: 4.5 }},
    {{ name: "Warehouse Way", target: 1200, stars: 16, bugs: 9, length: 2000, speed: 5 }},
    {{ name: "Dataflow Drive", target: 1600, stars: 18, bugs: 11, length: 2200, speed: 5.5 }},
    {{ name: "Notebook Narrows", target: 2000, stars: 20, bugs: 14, length: 2500, speed: 6 }},
    {{ name: "Eventhouse Express", target: 2500, stars: 22, bugs: 17, length: 2800, speed: 6.5 }},
    {{ name: "Shortcut Sprint", target: 3000, stars: 24, bugs: 20, length: 3000, speed: 7 }},
    {{ name: "Capacity Canyon", target: 3500, stars: 26, bugs: 24, length: 3300, speed: 7.5 }},
    {{ name: "OneLake Overdrive", target: 4000, stars: 28, bugs: 28, length: 3600, speed: 8 }},
    {{ name: "Spark Summit", target: 5000, stars: 32, bugs: 32, length: 4000, speed: 9 }}
];

const canvas = document.getElementById("gameCanvas");
const ctx = canvas.getContext("2d");
const W = 600, H = 500;

let gameRunning = false, currentLevel = 0, levelScore = 0, totalScore = 0, multiplier = 1;
let consecutiveStars = 0, distance = 0, raceFinished = false, lives = 3;
let car = {{ x: W/2, y: H - 80, width: 40, height: 60, speed: 0 }};
let stars = [], bugs = [], roadOffset = 0, keys = {{}}, floatingTexts = [];

const sessionId = "{session_id}";
const playerName = "{PLAYER_NAME}";

function recordEvent(eventType, extraData = {{}}) {{
    const event = {{
        EventId: crypto.randomUUID(),
        Timestamp: new Date().toISOString(),
        SessionId: sessionId,
        PlayerId: playerName,
        EventType: eventType,
        Level: currentLevel + 1,
        LevelName: LEVELS[currentLevel]?.name || "Unknown",
        Score: levelScore,
        TotalScore: totalScore,
        Lives: lives,
        Multiplier: multiplier,
        ...extraData
    }};
    window.gameEvents.push(event);
    window.dispatchEvent(new CustomEvent('gev', {{detail: event}}));
    document.getElementById("event-count").textContent = "Events: " + window.gameEvents.length;
}}

function updateLivesDisplay() {{ document.getElementById("lives").textContent = "❤️".repeat(lives) + "🖤".repeat(3-lives); }}

function showFloatingText(text, x, y, color) {{
    floatingTexts.push({{ text: text, x: x, y: y, color: color, life: 40 }});
}}

function drawFloatingTexts() {{
    floatingTexts = floatingTexts.filter(ft => {{
        ctx.save();
        ctx.globalAlpha = Math.max(0, ft.life / 40);
        ctx.fillStyle = ft.color;
        ctx.font = "bold 22px Arial";
        ctx.textAlign = "center";
        ctx.strokeStyle = "rgba(0,0,0,0.8)";
        ctx.lineWidth = 3;
        ctx.strokeText(ft.text, ft.x, ft.y);
        ctx.fillText(ft.text, ft.x, ft.y);
        ctx.restore();
        ft.y -= 1.2; ft.life--;
        return ft.life > 0;
    }});
}}

function initLevel() {{
    const level = LEVELS[currentLevel];
    distance = 0; levelScore = 0; multiplier = 1; consecutiveStars = 0; raceFinished = false;
    car.x = W/2; car.speed = level.speed;
    stars = []; bugs = []; floatingTexts = [];

    for (let i = 0; i < level.stars; i++) {{
        stars.push({{ x: 120 + Math.random() * 360, y: -(200 + i * (level.length/level.stars)), collected: false }});
    }}
    for (let i = 0; i < level.bugs; i++) {{
        bugs.push({{ x: 120 + Math.random() * 360, y: -(250 + i * (level.length/level.bugs)), hit: false }});
    }}

    document.getElementById("level-info").style.display = "none";
    updateLivesDisplay();
    updateHUD();
    recordEvent("LevelStart", {{ TargetScore: level.target }});
}}

function updateHUD() {{
    const level = LEVELS[currentLevel];
    document.getElementById("level").textContent = currentLevel + 1;
    document.getElementById("level-name").textContent = level.name;
    document.getElementById("score").textContent = levelScore;
    document.getElementById("multiplier").textContent = "x" + multiplier;
    document.getElementById("target").textContent = level.target;
    const progress = Math.min(100, (distance / level.length) * 100);
    document.getElementById("progress-fill").style.height = progress + "%";
    document.getElementById("progress-car").style.bottom = "calc(" + progress + "% - 10px)";
}}

function drawRoad() {{
    ctx.fillStyle = "#1a1a3e"; ctx.fillRect(0, 0, W, H);
    ctx.fillStyle = "#2d3436"; ctx.fillRect(100, 0, W-200, H);
    ctx.fillStyle = "#3498db"; ctx.fillRect(95, 0, 8, H); ctx.fillRect(W-103, 0, 8, H);
    ctx.strokeStyle = "rgba(255,255,255,0.4)"; ctx.setLineDash([40, 25]); ctx.lineWidth = 4;
    ctx.beginPath();
    for (let y = (roadOffset % 65) - 65; y < H + 65; y += 65) {{ ctx.moveTo(W/2, y); ctx.lineTo(W/2, y + 40); }}
    ctx.stroke(); ctx.setLineDash([]);

    const finishY = H - (distance - (LEVELS[currentLevel].length - 100));
    if (finishY > -50 && finishY < H + 50) {{
        for (let x = 100; x < W - 100; x += 20) {{
            for (let row = 0; row < 2; row++) {{
                ctx.fillStyle = ((x/20) + row) % 2 === 0 ? "#fff" : "#000";
                ctx.fillRect(x, finishY + row*20, 20, 20);
            }}
        }}
        ctx.fillStyle = "#ffd700"; ctx.font = "bold 22px Arial"; ctx.textAlign = "center";
        ctx.fillText("🏁 FINISH 🏁", W/2, finishY - 15); ctx.textAlign = "left";
    }}
}}

function drawCar() {{
    ctx.save(); ctx.translate(car.x, car.y);
    ctx.fillStyle = "#e74c3c";
    ctx.beginPath(); ctx.roundRect(-car.width/2, -car.height/2, car.width, car.height, 8); ctx.fill();
    ctx.fillStyle = "#74b9ff"; ctx.fillRect(-14, -car.height/2 + 8, 28, 16);
    ctx.fillStyle = "#fff"; ctx.fillRect(-3, -car.height/2, 6, car.height);
    ctx.fillStyle = "#222";
    ctx.fillRect(-car.width/2 - 4, -car.height/2 + 8, 6, 16);
    ctx.fillRect(car.width/2 - 2, -car.height/2 + 8, 6, 16);
    ctx.fillRect(-car.width/2 - 4, car.height/2 - 24, 6, 16);
    ctx.fillRect(car.width/2 - 2, car.height/2 - 24, 6, 16);
    ctx.restore();
}}

function drawStars() {{
    stars.forEach(star => {{
        if (!star.collected) {{
            const screenY = star.y + distance;
            if (screenY > -40 && screenY < H + 40) {{ ctx.font = "32px Arial"; ctx.fillText("⭐", star.x - 16, screenY + 12); }}
        }}
    }});
}}

function drawBugs() {{
    bugs.forEach(bug => {{
        if (!bug.hit) {{
            const screenY = bug.y + distance;
            if (screenY > -40 && screenY < H + 40) {{ ctx.font = "30px Arial"; ctx.fillText("🐛", bug.x - 15, screenY + 10); }}
        }}
    }});
}}

function checkCollisions() {{
    if (raceFinished) return;
    stars.forEach(star => {{
        if (!star.collected) {{
            const screenY = star.y + distance;
            if (Math.abs(car.x - star.x) < 40 && Math.abs(car.y - screenY) < 45) {{
                star.collected = true; consecutiveStars++;
                const prevMult = multiplier;
                multiplier = Math.min(10, Math.floor(consecutiveStars / 3) + 1);
                const points = 50 * multiplier; levelScore += points;
                showFloatingText("+" + points + (multiplier > 1 ? " (x" + multiplier + ")" : ""), star.x, screenY, "#ffd700");
                if (multiplier > prevMult) {{
                    showFloatingText("COMBO x" + multiplier + "!", W/2, H/2 - 40, "#00e5ff");
                }}
                recordEvent("StarCollected", {{ Points: points }});
            }}
        }}
    }});
    bugs.forEach(bug => {{
        if (!bug.hit) {{
            const screenY = bug.y + distance;
            if (Math.abs(car.x - bug.x) < 35 && Math.abs(car.y - screenY) < 40) {{
                bug.hit = true; levelScore = Math.max(0, levelScore - 30);
                consecutiveStars = 0; multiplier = 1;
                showFloatingText("-30", bug.x, screenY, "#ff4757");
                recordEvent("BugHit", {{ PenaltyPoints: 30 }});
            }}
        }}
    }});
    updateHUD();
}}

function checkFinish() {{
    const level = LEVELS[currentLevel];
    if (!raceFinished && distance >= level.length) {{
        raceFinished = true; gameRunning = false;
        const success = levelScore >= level.target;
        const levelInfo = document.getElementById("level-info");

        if (success) {{
            totalScore += levelScore;
            recordEvent("LevelComplete", {{ Success: true }});
            if (currentLevel >= LEVELS.length - 1) {{
                document.getElementById("level-title").textContent = "🏆 CHAMPION! 🏆";
                document.getElementById("level-message").innerHTML = "All tracks done!<br>Final Score: <b style='color:#ffd700;'>" + totalScore + "</b>";
                recordEvent("GameComplete", {{ FinalScore: totalScore }});
                document.getElementById("continue-btn").style.display = "none";
                window.dispatchEvent(new CustomEvent('gend', {{detail: {{ev: window.gameEvents}}}}));
                saveScore();
            }} else {{
                document.getElementById("level-title").textContent = "✅ Level Complete!";
                document.getElementById("level-message").innerHTML = "Score: <b style='color:#2ecc71;'>" + levelScore + "</b> / Target: <b style='color:#ffd700;'>" + level.target + "</b><br>Total: <b>" + (totalScore + levelScore) + "</b>";
            }}
            levelInfo.style.display = "block";
        }} else {{
            lives--; updateLivesDisplay();
            recordEvent("LevelFailed", {{ Success: false }});
            const missing = level.target - levelScore;
            if (lives <= 0) {{
                document.getElementById("game-over").innerHTML =
                    '<h2 style="color:#e74c3c;">💀 GAME OVER 💀</h2>'
                    + '<p style="margin-top:10px;">Last level: <b>' + level.name + '</b></p>'
                    + '<p>Score: <b style="color:#e74c3c;">' + levelScore + '</b> / Target: <b style="color:#ffd700;">' + level.target + '</b></p>'
                    + '<p>Missing: <b style="color:#e74c3c;">' + missing + ' pts</b></p>'
                    + '<p style="margin-top:15px;">Final Score: <b style="color:#ffd700;font-size:22px;">' + totalScore + '</b></p>'
                    + '<p>Reached Level: <span>' + (currentLevel + 1) + '</span></p>'
                    + '<p style="margin-top:10px; color:#2ecc71;">✅ Telemetria inviata!</p>'
                    + '<button onclick="restartGame()" style="margin-top:15px; padding:12px 30px; background:#e74c3c; color:white; border:none; border-radius:10px; cursor:pointer;">🔄 Play Again</button>';
                document.getElementById("game-over").style.display = "block";
                recordEvent("GameOver", {{ FinalScore: totalScore }});
                window.dispatchEvent(new CustomEvent('gend', {{detail: {{ev: window.gameEvents}}}}));
                saveScore();
            }} else {{
                document.getElementById("level-title").textContent = "❌ Try Again!";
                document.getElementById("level-message").innerHTML =
                    "Score: <b style='color:#e74c3c;'>" + levelScore + "</b> / Target: <b style='color:#ffd700;'>" + level.target + "</b>"
                    + "<br>Missing: <b style='color:#e74c3c;'>" + missing + " pts</b>"
                    + "<br>Lives left: <b style='color:#ff6b6b;'>" + "❤️".repeat(lives) + "</b>";
                levelInfo.style.display = "block";
            }}
        }}
    }}
}}

function gameLoop() {{
    if (!gameRunning) return;
    if (keys.ArrowLeft || keys.KeyA) car.x = Math.max(130, car.x - 7);
    if (keys.ArrowRight || keys.KeyD) car.x = Math.min(W - 130, car.x + 7);
    distance += car.speed;
    roadOffset += car.speed;
    drawRoad(); drawStars(); drawBugs(); drawCar(); drawFloatingTexts();
    checkCollisions(); checkFinish();
    requestAnimationFrame(gameLoop);
}}

function startGame() {{
    document.getElementById("start-btn").style.display = "none";
    document.getElementById("game-over").style.display = "none";
    window.gameEvents = [];
    currentLevel = 0; totalScore = 0; lives = 3;
    recordEvent("GameStart");
    initLevel();
    gameRunning = true;
    document.getElementById('game-wrapper').focus();
    gameLoop();
}}

function continueGame() {{
    const level = LEVELS[currentLevel];
    if (levelScore >= level.target) currentLevel++;
    initLevel();
    gameRunning = true;
    document.getElementById('game-wrapper').focus();
    gameLoop();
}}

function restartGame() {{
    document.getElementById("game-over").style.display = "none";
    startGame();
}}

function saveScore() {{
    const scores = JSON.parse(localStorage.getItem('fabricRacingScores') || '[]');
    scores.push({{ name: playerName, score: totalScore, level: currentLevel + 1, date: new Date().toLocaleDateString() }});
    scores.sort((a, b) => b.score - a.score);
    localStorage.setItem('fabricRacingScores', JSON.stringify(scores.slice(0, 10)));
    loadLeaderboard();
}}

function loadLeaderboard() {{
    const scores = JSON.parse(localStorage.getItem('fabricRacingScores') || '[]');
    const el = document.getElementById('leaderboard-list');
    if (!el) return;
    el.innerHTML = scores.length
        ? scores.map((s, i) => '<li><span>' + (i+1) + '. ' + s.name + ' <span style="color:#aaa;font-size:11px;">(Lv.' + s.level + ' • ' + s.date + ')</span></span><b style="color:#ffd700;">' + s.score + ' pts</b></li>').join('')
        : '<li style="color:#888;">No scores yet — be the first!</li>';
}}
loadLeaderboard();

const wrapper = document.getElementById('game-wrapper');
function _kd(e) {{ keys[e.code] = true; if (["ArrowLeft","ArrowRight","ArrowUp","ArrowDown"].includes(e.key)) e.preventDefault(); }}
function _ku(e) {{ keys[e.code] = false; }}
document.addEventListener("keydown", _kd);
document.addEventListener("keyup", _ku);
wrapper.addEventListener("keydown", _kd);
wrapper.addEventListener("keyup", _ku);
wrapper.addEventListener("click", () => wrapper.focus());
</script>
</body>
</html>
'''

display(HTML(game_html + bridge_js))
print("🎮 Use ← → arrow keys to steer. Collect ⭐, avoid 🐛!")
print("📡 Telemetria inviata automaticamente ogni 2 secondi")


In [ ]:
# 📊 CELL 3 (OPZIONALE): Verifica Telemetria in Eventhouse
# Esegui questa cella per vedere i tuoi eventi nel KQL Database

# Configura l'URI del cluster (lo trovi in Eventhouse → KQL Database → Properties)
CLUSTER_URI = "https://trd-XXXXX.kusto.fabric.microsoft.com"  # <-- AGGIORNA
DB_NAME = "RacingGameEventhouse"  # Nome del tuo KQL Database

import requests

# Ottieni token Kusto
try:
    import notebookutils as _nu
    _ktok = _nu.credentials.getToken("https://kusto.kusto.windows.net")
except:
    import subprocess as _sp
    _ktok = _sp.run("az account get-access-token --resource https://kusto.kusto.windows.net --query accessToken -o tsv",
                     capture_output=True, text=True, shell=True).stdout.strip()

# Query: ultimi eventi
kql = """GameEvents 
| where Timestamp > ago(1h) 
| summarize Eventi=count(), MaxScore=max(Score), MaxLevel=max(Level)
    by PlayerId, SessionId 
| order by MaxScore desc
| take 10"""

r = requests.post(f"{CLUSTER_URI}/v1/rest/query",
    headers={"Authorization": f"Bearer {_ktok}", "Content-Type": "application/json"},
    json={"db": DB_NAME, "csl": kql}, timeout=10)

if r.status_code == 200:
    data = r.json()
    cols = [c['ColumnName'] for c in data['Tables'][0]['Columns']]
    rows = data['Tables'][0]['Rows']
    print(f"📊 {len(rows)} sessioni nell'ultima ora:")
    print("  " + " | ".join(cols))
    for row in rows: 
        print("  " + " | ".join(str(x) for x in row))
else: 
    print(f"⚠️ Errore query: {r.status_code}")
    print("Verifica CLUSTER_URI e DB_NAME")